# Binary Calibration Notebook

This notebook computes **Brier Score** and **Expected Calibration Error (ECE)** for the four binary runs:

1. `ben_sarc_binary` — `banglabert`
2. `ben_sarc_binary` — `banglabert_fgm`
3. `banglasarc3_binary` — `banglabert`
4. `banglasarc3_binary` — `banglabert_fgm`

It automatically resolves the correct `checkpoint-*` subfolder inside each checkpoint root.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import brier_score_loss

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SPLITS = Path("../01_data/interim/splits")
CHECKPOINTS = Path("../03_models/checkpoints")
TABLES = Path("../04_outputs/tables")

MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128
BATCH_SIZE = 8

TABLES.mkdir(parents=True, exist_ok=True)

print("Splits:", SPLITS.resolve())
print("Checkpoints:", CHECKPOINTS.resolve())
print("Tables:", TABLES.resolve())

Splits: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detection/01_data/interim/splits
Checkpoints: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detection/03_models/checkpoints
Tables: /Users/sefayet/Desktop/Github/Machine_Learning-Deep_Learning-Courses-and-Paper-Publish/Thesis_Papers/Sarcasm_detection/04_outputs/tables


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [4]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

In [5]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if np.sum(mask) == 0:
            continue

        bin_conf = np.mean(y_prob[mask])
        bin_acc = np.mean(y_true[mask])
        ece += (np.sum(mask) / len(y_true)) * abs(bin_acc - bin_conf)

    return float(ece)

In [6]:
def load_test_dataset(split_file):
    df = pd.read_csv(split_file)
    df = df[["text", "label_binary"]].rename(columns={"label_binary": "label"})

    ds = Dataset.from_pandas(df, preserve_index=False)
    ds = ds.map(tokenize_batch, batched=True)
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")

    return df, ds

In [7]:
def resolve_checkpoint_dir(checkpoint_root):
    checkpoint_root = Path(checkpoint_root)

    # Case 1: root itself is a real checkpoint
    if (checkpoint_root / "config.json").exists():
        return checkpoint_root

    # Case 2: find checkpoint-* subfolders
    ckpts = sorted(
        [p for p in checkpoint_root.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[-1])
    )

    if not ckpts:
        raise FileNotFoundError(f"No checkpoint-* folder found inside: {checkpoint_root}")

    # Prefer trainer_state best_model_checkpoint if available at the root
    trainer_state_file = checkpoint_root / "trainer_state.json"
    if trainer_state_file.exists():
        with open(trainer_state_file, "r", encoding="utf-8") as f:
            trainer_state = json.load(f)
        best_ckpt = trainer_state.get("best_model_checkpoint", None)
        if best_ckpt:
            best_ckpt = Path(best_ckpt)
            if best_ckpt.exists():
                return best_ckpt

    # Fallback: latest checkpoint
    return ckpts[-1]

In [8]:
def get_binary_probs(checkpoint_dir, split_file):
    df, ds = load_test_dataset(split_file)

    resolved_dir = resolve_checkpoint_dir(checkpoint_dir)
    print("Loading checkpoint from:", resolved_dir)

    model = AutoModelForSequenceClassification.from_pretrained(resolved_dir)
    trainer = Trainer(model=model)

    output = trainer.predict(ds)
    logits = output.predictions

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    prob_pos = probs[:, 1]

    y_true = df["label"].to_numpy()

    return y_true, prob_pos

In [9]:
runs = [
    {
        "dataset": "ben_sarc_binary",
        "model": "banglabert",
        "checkpoint_dir": "../03_models/checkpoints/banglabert_ben_sarc_binary",
        "split_file": "../01_data/interim/splits/ben_sarc_binary_test.csv",
    },
    {
        "dataset": "ben_sarc_binary",
        "model": "banglabert_fgm",
        "checkpoint_dir": "../03_models/checkpoints/banglabert_fgm_ben_sarc_binary",
        "split_file": "../01_data/interim/splits/ben_sarc_binary_test.csv",
    },
    {
        "dataset": "banglasarc3_binary",
        "model": "banglabert",
        "checkpoint_dir": "../03_models/checkpoints/banglabert_banglasarc3_binary",
        "split_file": "../01_data/interim/splits/banglasarc3_binary_test.csv",
    },
    {
        "dataset": "banglasarc3_binary",
        "model": "banglabert_fgm",
        "checkpoint_dir": "../03_models/checkpoints/banglabert_fgm_banglasarc3_binary",
        "split_file": "../01_data/interim/splits/banglasarc3_binary_test.csv",
    },
]

In [10]:
rows = []

for run in runs:
    y_true, prob_pos = get_binary_probs(run["checkpoint_dir"], run["split_file"])

    brier = brier_score_loss(y_true, prob_pos)
    ece = expected_calibration_error(y_true, prob_pos, n_bins=10)

    rows.append({
        "dataset": run["dataset"],
        "model": run["model"],
        "brier_score": brier,
        "ece": ece,
    })

calibration_df = pd.DataFrame(rows)
calibration_df

Map: 100%|██████████| 2564/2564 [00:00<00:00, 13754.42 examples/s]


Loading checkpoint from: ../03_models/checkpoints/banglabert_ben_sarc_binary/checkpoint-5128


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8883.99it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Map: 100%|██████████| 2564/2564 [00:00<00:00, 26680.85 examples/s]


Loading checkpoint from: ../03_models/checkpoints/banglabert_fgm_ben_sarc_binary/checkpoint-5128


...

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8281.89it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Map: 100%|██████████| 802/802 [00:00<00:00, 17220.66 examples/s]


Loading checkpoint from: ../03_models/checkpoints/banglabert_fgm_banglasarc3_binary/checkpoint-1604


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8012.99it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


,dataset,model,brier_score,ece
0,ben_sarc_binary,banglabert,0.168705,0.135248
1,ben_sarc_binary,banglabert_fgm,0.138111,0.046411
2,banglasarc3_binary,banglabert,0.190553,0.131891
3,banglasarc3_binary,banglabert_fgm,0.169369,0.052378


In [11]:
calibration_df.to_csv(TABLES / "binary_calibration_results.csv", index=False)
print("Saved:", TABLES / "binary_calibration_results.csv")

Saved: ../04_outputs/tables/binary_calibration_results.csv


In [12]:
pivot_brier = calibration_df.pivot(index="dataset", columns="model", values="brier_score").reset_index()
pivot_ece = calibration_df.pivot(index="dataset", columns="model", values="ece").reset_index()

pivot_brier["brier_gain_fgm"] = pivot_brier["banglabert"] - pivot_brier["banglabert_fgm"]
pivot_ece["ece_gain_fgm"] = pivot_ece["banglabert"] - pivot_ece["banglabert_fgm"]

print("Brier comparison")
display(pivot_brier)

print("ECE comparison")
display(pivot_ece)

Brier comparison


model,dataset,banglabert,banglabert_fgm,brier_gain_fgm
0,banglasarc3_binary,0.190553,0.169369,0.021184
1,ben_sarc_binary,0.168705,0.138111,0.030594


ECE comparison


model,dataset,banglabert,banglabert_fgm,ece_gain_fgm
0,banglasarc3_binary,0.131891,0.052378,0.079513
1,ben_sarc_binary,0.135248,0.046411,0.088837
